# M2 — Localização de Facilidades

**Treinamento de Otimização Aplicada — Genoa para Gradus**

## Caso: Beerlink expande — onde abrir os CDs?

Maurício cresceu. A Beerlink hoje atende **10 cidades** em Grande SP, ABC, Vale do Paraíba e interior. Está pagando frete avulso a CDs alugados mês a mês. Quer decidir, de forma estratégica, em quais cidades operar CDs próprios.

**5 sites candidatos** a CD. Cada um tem:
- Aluguel mensal (custo fixo de abrir)
- Capacidade em caixas/mês

**Frete:** R\$ 0,80 / cx / km, proporcional à distância CD → cidade.

Decisão dupla:
- $y_j \in \{0,1\}$: abrir o CD candidato $j$?
- $x_{ij} \ge 0$: fluxo (caixas/mês) do CD $j$ para a cidade $i$

É **Capacitated Facility Location Problem (CFLP)** — um dos MIPs clássicos.

> ⚠️ **Versão TEMPLATE** — algumas células contêm `# TODO` para você preencher. Se ficar travado, abra a versão solução: `m2_localizacao_solution.ipynb` (ou o `.py` em `scripts/`). Se estiver no Colab, o link da solução está no deck.

## Setup

In [ ]:
%pip install -q ortools gurobipy

In [ ]:
import math
import pandas as pd

# 5 candidatos a CD
CANDIDATOS = ['SP-Pinheiros', 'Guarulhos', 'Campinas', 'Sorocaba', 'SJCampos']
ALUGUEL = {'SP-Pinheiros': 80, 'Guarulhos': 55, 'Campinas': 45, 'Sorocaba': 38, 'SJCampos': 42}  # R$ mil/mês
CAPAC   = {'SP-Pinheiros': 8000, 'Guarulhos': 6500, 'Campinas': 5500, 'Sorocaba': 4800, 'SJCampos': 5200}  # cx/mês

# 10 cidades atendidas
CIDADES = ['SP-Capital', 'Guarulhos', 'Osasco', 'ABC', 'Campinas',
           'Sorocaba', 'Jundiaí', 'SJCampos', 'Taubaté', 'Piracicaba']
DEMANDA = {'SP-Capital': 3500, 'Guarulhos': 1100, 'Osasco': 900, 'ABC': 1200, 'Campinas': 1500,
           'Sorocaba': 900, 'Jundiaí': 600, 'SJCampos': 1100, 'Taubaté': 600, 'Piracicaba': 500}

# Coordenadas aproximadas (lat, lon) — usadas só para calcular distâncias
COORDS = {
    'SP-Pinheiros': (-23.567, -46.685), 'Guarulhos': (-23.463, -46.533),
    'Campinas': (-22.907, -47.063),     'Sorocaba': (-23.501, -47.458),
    'SJCampos': (-23.179, -45.886),
    'SP-Capital': (-23.550, -46.633),   'Osasco': (-23.532, -46.792),
    'ABC': (-23.660, -46.561),          'Jundiaí': (-23.186, -46.884),
    'Taubaté': (-23.026, -45.555),      'Piracicaba': (-22.725, -47.649),
}

def km(a, b): return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111

TARIFA_KM = 0.80   # R$ / cx / km
FRETE = {(c, cd): km(COORDS[c], COORDS[cd]) * TARIFA_KM for c in CIDADES for cd in CANDIDATOS}

print(f'Demanda total: {sum(DEMANDA.values()):,} cx/mês')
print(f'Capacidade total (se todos os 5 CDs abrirem): {sum(CAPAC.values()):,} cx/mês')
print(f'Mín. {(sum(DEMANDA.values())+max(CAPAC.values())-1)//max(CAPAC.values())} CD(s) tecnicamente bastaria(m) — mas frete e dispersão geográfica forçam mais.')

## Modelo CFLP — formulação

$$\min \sum_j r_j y_j + \sum_{i,j} c_{ij} x_{ij}$$

Sujeito a:
- **Demanda:** $\sum_j x_{ij} = d_i \quad \forall i$
- **Capacidade (só se aberto):** $\sum_i x_{ij} \le Q_j\, y_j \quad \forall j$
- **Domínios:** $y_j \in \{0,1\},\ x_{ij} \ge 0$

O "truque" da multiplicação $Q_j \cdot y_j$ é o padrão clássico: se $y_j = 0$, o lado direito é zero (ninguém pode enviar nada). Se $y_j = 1$, vira a capacidade real.

### Solver 1 — OR-Tools (CBC)

In [ ]:
from ortools.linear_solver import pywraplp

def solve_cflp_ortools(cidades, sites, demanda, capacidade, aluguel, frete):
    """CFLP: decida quais CDs abrir (binaria y_j) + quanto enviar (x_ij contínua)."""
    s = pywraplp.Solver.CreateSolver('CBC')
    I, J = range(len(cidades)), range(len(sites))

    # TODO: variaveis
    # y[j] in {0,1}: CD j aberto
    # x[i,j] >= 0: caixas enviadas de j para a cidade i (continua)

    # TODO: restricoes
    # (1) demanda atendida: sum_j x[i,j] == demanda[i] forall i
    # (2) capacidade (so se aberto): sum_i x[i,j] <= capacidade[j] * y[j]

    # TODO: FO + solve
    # minimizar sum_j aluguel[j]*y[j] + sum_{ij} frete[i][j]*x[i,j]

    raise NotImplementedError("Complete o CFLP OR-Tools")


### Solver 2 — Gurobi (gurobipy)

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_cflp_gurobi(cidades, sites, demanda, capacidade, aluguel, frete):
    """Mesmo CFLP em Gurobi. Use tupledict idioms (y.sum(), x.sum(i,'*'))."""
    m = gp.Model('cflp')
    m.Params.OutputFlag = 0

    # TODO: variaveis (m.addVars com indices)
    # TODO: restricoes (m.addConstrs com generator + x.sum(i,'*'))
    # TODO: FO + m.optimize()

    raise NotImplementedError("Complete o CFLP Gurobi")


### Lendo a solução

Os dois solvers convergem para o mesmo ótimo. **4 CDs abertos** (Guarulhos fica fora), custo total ≈ R\$ 330 mil/mês.

Por que Guarulhos não é aberto?
- SP-Pinheiros está perto e tem capacidade ampla (8000 cx)
- Pode atender SP-Capital + Guarulhos + Osasco + ABC numa só facility
- Abrir Guarulhos custaria R\$ 55k/mês de aluguel para economizar pouco frete

**Insight clássico de localização:** o solver consolida demanda em facilidades centrais (alta capacidade, próximas à maior demanda) mesmo que isso custe mais frete — porque o aluguel fixo de uma facility extra raramente compensa.

---

## Exercícios de extensão

### 1. Multi-produto (IPA + Pilsen com capacidades separadas)

Cada CD agora tem **capacidade dividida**: até 60 % para IPA, 40 % para Pilsen. Demanda também separada por produto.

Variáveis: $x_{ij}^{p}$ para cada produto $p$. Capacidade vira por produto: $\sum_i x_{ij}^p \le 0{,}6\, Q_j\, y_j$ (IPA) e $\le 0{,}4\, Q_j\, y_j$ (Pilsen).

In [ ]:
# Demanda separada por produto (60% IPA, 40% Pilsen). Modelo CFLP multi-produto.

# TODO: estenda o CFLP para multi-produto
# - x[i,j,p] >= 0 para produto p in {IPA, Pilsen}
# - capacidade do CD ainda eh global (sum_p sum_i x[i,j,p] <= Q*y[j])
# - demanda separada por produto: sum_j x[i,j,p] == d[i,p]

raise NotImplementedError("Complete a extensao multi-produto")


### 2. Redundância — cada cidade atendida por ≥ 2 CDs (resiliência)

Restrição extra: $\sum_j \mathbb{1}[x_{ij} > 0] \ge 2$ para cada cidade. Para linearizar, usamos uma variável auxiliar $z_{ij} \in \{0,1\}$ indicando se a cidade $i$ é atendida pelo CD $j$, ligada a $x_{ij}$ via big-M.

In [ ]:
def solve_cflp_redundancia():
    """Cada cidade deve ser atendida por >= 2 CDs abertos (resiliencia operacional)."""

    # TODO: adicione variavel binaria z[i,j] = 1 se a cidade i tem CD j na lista de fallback
    # TODO: restricao: sum_j z[i,j] >= 2 forall i
    # TODO: z[i,j] <= y[j] (so CDs abertos contam)

    raise NotImplementedError("Complete o CFLP com redundancia")


### 3. p-median na escala — 20 cidades, 8 candidatos, exatamente p=3 CDs

Geramos uma instância maior e medimos tempo de OR-Tools vs Gurobi. Aqui o solver comercial costuma ganhar de fato:

In [ ]:
import random
random.seed(42)

# 20 cidades, 8 candidatos, exatamente p=3 CDs
P = 3

# TODO: monte o p-median.
# - mesma estrutura do CFLP mas com restricao adicional sum_j y[j] == P
# - rode no OR-Tools E no Gurobi; compare tempo e resultado.

raise NotImplementedError("Complete o p-median")


## Lições do M2

1. **CFLP é um MIP "limpo"** — binárias e contínuas, sem big-M no caso simples. Pareia bem com Gurobi.
2. **A multiplicação $Q_j \cdot y_j$** é o padrão para "ligar fluxo a binária de abrir". Memorize.
3. **Redundância é cara** — exigir 2 CDs por cidade aumenta significativamente o custo, mas é decisão estratégica (resiliência).
4. **Em escala** (20+ cidades, 10+ candidatos), o solver comercial Gurobi começa a fazer diferença mensurável. É o regime de problemas reais.

Conexão com o M1: lá decidimos rotas com CD fixo. Aqui decidimos o CD em si. Em consultoria, os dois problemas se combinam — "localização + roteamento" é o problema de design de rede de distribuição completo (e o terreno onde Gurobi + lazy constraints é estado da arte).